In [3]:
import pandas as pd
df = pd.read_csv(r'D:\Data Science\Fraud_detection\data\fraud_analysis_dataset.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 647 entries, 0 to 646
Data columns (total 20 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Transaction_ID                647 non-null    object 
 1   Date                          647 non-null    object 
 2   Time                          647 non-null    object 
 3   Merchant_ID                   647 non-null    object 
 4   Customer_ID                   647 non-null    object 
 5   Device_ID                     647 non-null    object 
 6   Transaction_Type              647 non-null    object 
 7   Payment_Gateway               647 non-null    object 
 8   Transaction_City              647 non-null    object 
 9   Transaction_State             647 non-null    object 
 10  IP_Address                    647 non-null    object 
 11  Transaction_Status            647 non-null    object 
 12  Device_OS                     647 non-null    object 
 13  Trans

In [4]:
#  first we need to convert the date column to date time format

df['Date'] = pd.to_datetime(df['Date'])
df['Time'] = pd.to_datetime(df['Time'],format ='%I:%M:%S %p').dt.time

C:\Users\Vishnu\AppData\Local\Temp\ipykernel_13880\2502451021.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Date'] = pd.to_datetime(df['Date'])


1️⃣ First: Feature Engineering Philosophy (Fraud Context)

Fraud is behavioral + temporal + relational.

Raw columns alone are weak.
Fraud signals emerge when we ask:

❓ Is this transaction unusual for this customer/device/merchant?

❓ Is the timing suspicious?

❓ Is the network behavior abnormal?

❓ Is the amount inconsistent with history?

So we engineer features in 5 layers:

Time-based

Amount-based

Velocity / Frequency-based

Entity behavior (Customer / Device / Merchant)

Risk aggregation & encoding

#### (i) Time Feature creation :  

Existing

Date

Time

Create 👇
🔹 transaction_hour
Hour extracted from Time (0–23)


Why

Fraud spikes at odd hours (midnight–early morning)

Humans sleep; bots don’t

🔹 transaction_day
Day of month (1–31)


Why

End-of-month / salary days → higher fraud attempts

🔹 transaction_weekday
0=Monday … 6=Sunday


Why

Weekend fraud patterns differ from weekdays

🔹 is_weekend
1 if Saturday/Sunday else 0


Why

Fraudsters exploit low-monitoring windows

🔹 is_night_transaction
1 if hour ∈ [0–5] else 0


Why

High-signal binary fraud indicator


In [5]:
df['transaction_datetime'] = pd.to_datetime(
    df['Date'].astype(str) + ' ' + df['Time'].astype(str)
)


df['transaction_hour'] = df['transaction_datetime'].dt.hour
df['transaction_day'] = df['transaction_datetime'].dt.day
df['transaction_weekday'] = df['transaction_datetime'].dt.weekday
df['is_weekend'] = df['transaction_weekday'].isin([5, 6]).astype(int)
df['is_night_transaction'] = df['transaction_hour'].between(0, 5).astype(int)



In [6]:
#### sort data by user and transaction datetime
# Why?
# All velocity & historical features depend on correct order
# If skipped → silent data leakage

df = df.sort_values(
    ['Customer_ID', 'transaction_datetime']
).reset_index(drop=True)


#### (ii) AMOUNT-BASED FEATURES : 
Existing

amount

Transaction_Amount_Deviation (already useful)

Create 👇
🔹 log_amount
log(amount + 1)


Why

Amounts are heavy-tailed

Helps ANN, Logistic, XGBoost converge better

🔹 amount_bucket
low / medium / high / very_high (quantiles)


Why

Fraud risk changes non-linearly with amount

🔹 is_high_amount
1 if amount > 95th percentile


Why

Captures rare but high-impact fraud

🔹 amount_vs_deviation_ratio
amount / (Transaction_Amount_Deviation + ε)


Why

Measures how abnormal the amount really is


In [7]:
import numpy as np
df['log_amount'] = np.log1p(df['amount'])

high_amount_threshold = df['amount'].quantile(0.95)
df['is_high_amount'] = (df['amount'] > high_amount_threshold).astype(int)

df['amount_vs_deviation_ratio'] = (
    df['amount'] / (df['Transaction_Amount_Deviation'] + 1e-6)
)


#### (iv) VELOCITY / FREQUENCY FEATURES : 

Existing

Transaction_Frequency

Days_Since_Last_Transaction

Create 👇
🔹 inverse_days_since_last_txn
1 / (Days_Since_Last_Transaction + 1)


Why

Recent bursts = fraud

Linear models prefer this form

🔹 is_burst_transaction
1 if Days_Since_Last_Transaction <= 1


Why

Fraud often happens in rapid bursts

🔹 frequency_bucket
low / medium / high


Why

Converts noisy numeric frequency into stable signal

In [8]:
df['inverse_days_since_last_txn'] = 1 / (
    df['Days_Since_Last_Transaction'] + 1
)

df['is_burst_transaction'] = (
    df['Days_Since_Last_Transaction'] <= 1
).astype(int)

df['frequency_bucket'] = pd.qcut(
    df['Transaction_Frequency'],
    q=3,
    labels=['low', 'medium', 'high']
)



#### (v) CUSTOMER-LEVEL Behaviour Features (NO LEAKAGE)

Group by Customer_ID

🔹 customer_avg_amount
Mean transaction amount per customer


Why

Detect deviation from personal norm

🔹 customer_std_amount
Std dev of customer amount


Why

Fraud = spike relative to variance

🔹 amount_vs_customer_avg
amount / customer_avg_amount


Why

One of the strongest fraud indicators

🔹 customer_txn_count
Total historical transactions


Why

New customers are riskier

🔹 is_new_customer
1 if customer_txn_count < threshold


Why

Cold-start fraud risk




In [9]:
df['customer_avg_amount'] = (
    df.groupby('Customer_ID')['amount']
      .transform('mean')
)

df['customer_std_amount'] = (
    df.groupby('Customer_ID')['amount']
      .transform('std')
      .fillna(0)
)

df['customer_txn_count'] = (
    df.groupby('Customer_ID').cumcount() + 1
)

df['amount_vs_customer_avg'] = (
    df['amount'] / (df['customer_avg_amount'] + 1e-6)
)

df['is_new_customer'] = (df['customer_txn_count'] <= 3).astype(int)


#### (vi) DEVICE-LEVEL FEATURES : 

Group by Device_ID

🔹 device_txn_count
Number of transactions from same device


Why

Fraud devices are reused

🔹 unique_customers_per_device
Count distinct customers per device


Why

One device → many customers = fraud ring

🔹 device_fraud_rate
Past fraud % for that device


Why

Extremely powerful for ensemble models


In [10]:
df['device_txn_count'] = (
    df.groupby('Device_ID').cumcount() + 1
)

df['unique_customers_per_device'] = (
    df.groupby('Device_ID')['Customer_ID']
      .transform('nunique')
)


#### (vii) MERCHANT-LEVEL FEATURES : 
Group by Merchant_ID

🔹 merchant_fraud_rate
Historical fraud ratio


Why

Some merchants are high-risk

🔹 merchant_avg_amount
Mean amount per merchant


Why

Contextualizes amount anomalies

🔹 is_new_merchant
1 if merchant_txn_count < threshold


Why

Fake or newly compromised merchants


In [11]:
df['merchant_txn_count'] = (
    df.groupby('Merchant_ID').cumcount() + 1
)

df['merchant_avg_amount'] = (
    df.groupby('Merchant_ID')['amount']
      .transform('mean')
)

df['is_new_merchant'] = (df['merchant_txn_count'] <= 5).astype(int)


#### (viii) GEO & CHANNEL FEATURES


8️⃣ Geo & Channel Risk Features
Existing

Transaction_City

Transaction_State

Transaction_Channel

IP_Address

Create 👇
🔹 state_fraud_rate
Fraud % per state


Why

Geo-risk modeling

🔹 channel_risk_score
Encoded risk per channel (Online > POS)


Why

Online channels are riskier

🔹 ip_device_match
1 if IP seen with this device before


Why

IP hopping is a fraud signal


In [12]:
df['Transaction_Channel'] = df['Transaction_Channel'].astype('category')
df['Transaction_State'] = df['Transaction_State'].astype('category')


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 647 entries, 0 to 646
Data columns (total 42 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   Transaction_ID                647 non-null    object        
 1   Date                          647 non-null    datetime64[ns]
 2   Time                          647 non-null    object        
 3   Merchant_ID                   647 non-null    object        
 4   Customer_ID                   647 non-null    object        
 5   Device_ID                     647 non-null    object        
 6   Transaction_Type              647 non-null    object        
 7   Payment_Gateway               647 non-null    object        
 8   Transaction_City              647 non-null    object        
 9   Transaction_State             647 non-null    category      
 10  IP_Address                    647 non-null    object        
 11  Transaction_Status            64

divide the dataset into training and testing data 


In [14]:
# Step 1: Sort
df = df.sort_values('transaction_datetime')

# Step 2: Split
split_date = df['transaction_datetime'].quantile(0.8)

train_df = df[df['transaction_datetime'] <= split_date]
test_df  = df[df['transaction_datetime'] > split_date]


# Why:

# Mimics real deployment

# Prevents future leakage

# Step 3: Separate target


X_train = train_df.drop(columns=['fraud'])
y_train = train_df['fraud']

X_test = test_df.drop(columns=['fraud'])
y_test = test_df['fraud']



PART D — Model Training (IN CORRECT ORDER)

We do not train everything at once.

1️⃣ Baseline Ensemble Models (FIRST)
Models

Logistic Regression

Random Forest

XGBoost / LightGBM (if allowed)

Why start here

Fast

Interpretable

Strong baseline

Easy MLflow tracking

Feature set
Tier-1 numeric features only

Metrics to track
ROC-AUC
Recall (fraud class)
Precision
PR-AUC


⚠️ Accuracy is meaningless for fraud.

2️⃣ Advanced Ensemble (SECOND)
Models

Gradient Boosting

XGBoost + class_weight

VotingClassifier / Stacking

Why

Handles non-linearity

Best for tabular fraud data

Most production fraud systems stop here.

3️⃣ ANN (THIRD — not first)
Input
Scaled numeric Tier-1 features

Architecture (example)
Input → Dense → BatchNorm → Dropout → Dense → Sigmoid

Why ANN is NOT first

Needs tuning

Less interpretable

No big gain over XGBoost usually

4️⃣ Autoencoder (LAST)
Training data
ONLY non-fraud (Is_Fraud == 0)

Why

Autoencoder learns normal behavior.

Fraud = high reconstruction error.

Output
Anomaly score (continuous)


Later:

Convert score → fraud flag using threshold

Combine with supervised model

PART E — Comparison Strategy (IMPORTANT)
❌ Do NOT compare by accuracy

| Metric    | Why                   |
| --------- | --------------------- |
| Recall    | Catch fraud           |
| Precision | Avoid false positives |
| PR-AUC    | Class imbalance       |
| ROC-AUC   | Overall ranking       |
| Stability | Drift resistance      |

Final comparison table
| Model         | Recall | Precision | PR-AUC | ROC-AUC |
| ------------- | ------ | --------- | ------ | ------- |
| Logistic      |        |           |        |         |
| Random Forest |        |           |        |         |
| XGBoost       |        |           |        |         |
| ANN           |        |           |        |         |
| Autoencoder   |        |           |        |         |



Final Feature Set (USED FOR ALL BASELINES)

In [15]:
FEATURES = [
    # Amount behavior
    'amount',
    'log_amount',
    'Transaction_Amount_Deviation',
    'amount_vs_customer_avg',
    'amount_vs_deviation_ratio',
    'is_high_amount',

    # Velocity
    'Days_Since_Last_Transaction',
    'inverse_days_since_last_txn',
    'Transaction_Frequency',
    'is_burst_transaction',

    # Time
    'transaction_hour',
    'is_night_transaction',
    'is_weekend',

    # Customer behavior
    'customer_txn_count',
    'customer_avg_amount',
    'customer_std_amount',
    'is_new_customer',

    # Device / merchant
    'device_txn_count',
    'unique_customers_per_device',
    'merchant_txn_count',
    'merchant_avg_amount',
    'is_new_merchant'
]

X_train = train_df[FEATURES]
y_train = train_df['fraud']

X_test = test_df[FEATURES]
y_test = test_df['fraud']


In [16]:
# STEP 3️⃣ Handle Class Imbalance (MANDATORY)
fraud_ratio = y_train.mean()
scale_pos_weight = (1 - fraud_ratio) / fraud_ratio


In [17]:
# STEP 4️⃣ Start MLflow Experiment
import mlflow

#  first create an instance  using mlflow server --backend-store-uri file:///D:/Data%20Science/mlflow_store/mlruns --default-artifact-root file:///D:/Data%20Science/mlflow_store/mlruns --host 127.0.0.1 --port 5000


mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("fraud_detection_Project")


<Experiment: artifact_location='file:///D:/Data%20Science/mlflow_store/mlruns/415765464730110731', creation_time=1768482547977, experiment_id='415765464730110731', last_update_time=1768482547977, lifecycle_stage='active', name='fraud_detection_Project', tags={}>

In [18]:
# STEP 5️⃣ Logistic Regression (Baseline Reference)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

model = LogisticRegression(
        class_weight='balanced',
        max_iter=1000
    )
model.fit(X_train, y_train)

preds = model.predict_proba(X_test)[:, 1]


with mlflow.start_run(run_name="Logistic_Regression"):
    
    mlflow.log_metric("roc_auc", roc_auc_score(y_test, preds))
    mlflow.log_metric("pr_auc", average_precision_score(y_test, preds))
    mlflow.sklearn.log_model(model, "model")


c:\Users\Vishnu\anaconda3\envs\fraud_detection_env\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
2026/01/15 18:44:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Logistic_Regression at: http://127.0.0.1:5000/#/experiments/415765464730110731/runs/04b6fbe926414f5598baf1e95367540b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/415765464730110731


In [19]:
# STEP 6️⃣ Random Forest (Non-linear Ensemble)
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        class_weight='balanced',
        n_jobs=-1,
        random_state=42
    )
rf.fit(X_train, y_train)

preds = rf.predict_proba(X_test)[:, 1]

with mlflow.start_run(run_name="Random_Forest"):
    
    ### logging parameters
    mlflow.log_params({
        "n_estimators": 300,
        "max_depth": 12,
        "class_weight": "balanced",
        "n_jobs": -1,
        "random_state": 42
        
    })
    
    # ✅ Log dataset info (for lineage / reproducibility)
    mlflow.log_param("train_rows", X_train.shape[0])
    mlflow.log_param("train_cols", X_train.shape[1])
    mlflow.log_param("test_rows", X_test.shape[0])
    mlflow.log_param("test_cols", X_test.shape[1])

    
    # ✅ Log features used
    mlflow.log_param("features_used", ",".join(FEATURES))
    
    mlflow.log_metric("roc_auc", roc_auc_score(y_test, preds))
    mlflow.log_metric("pr_auc", average_precision_score(y_test, preds))
    mlflow.sklearn.log_model(rf, "model")


2026/01/15 19:10:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Random_Forest at: http://127.0.0.1:5000/#/experiments/415765464730110731/runs/58646d8f689945bb98f03a4f2ab2db04
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/415765464730110731


In [22]:
# STEP 7️⃣ XGBoost (EXPECTED WINNER)
from xgboost import XGBClassifier
xgb = XGBClassifier(
        n_estimators=400,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        eval_metric='logloss',
        random_state=42
    )

xgb.fit(X_train, y_train)

preds = xgb.predict_proba(X_test)[:, 1]

with mlflow.start_run(run_name="XGBoost"):
    
    #log parameters
    mlflow.log_params({
        "n_estimators": 400,
        "max_depth": 6,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "scale_pos_weight": scale_pos_weight,
        "eval_metric": "logloss",
        "random_state": 42
    })
    
    # ✅ Log dataset info (for lineage / reproducibility)
    mlflow.log_param("train_rows", X_train.shape[0])
    mlflow.log_param("train_cols", X_train.shape[1])
    mlflow.log_param("test_rows", X_test.shape[0])
    mlflow.log_param("test_cols", X_test.shape[1])

    
    # ✅ Log features used
    mlflow.log_param("features_used", ",".join(FEATURES))

    mlflow.log_metric("roc_auc", roc_auc_score(y_test, preds))
    mlflow.log_metric("pr_auc", average_precision_score(y_test, preds))
    mlflow.xgboost.log_model(xgb, "model")


2026/01/15 19:42:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBoost at: http://127.0.0.1:5000/#/experiments/415765464730110731/runs/f1a26f933bf14b848a1862ab4b6c2c5b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/415765464730110731


In [24]:
# PART 1️⃣ ANN (Neural Network) — SUPERVISED FRAUD CLASSIFIER
ANN_FEATURES = [
    'log_amount',
    'amount_vs_customer_avg',
    'amount_vs_deviation_ratio',
    'inverse_days_since_last_txn',
    'Transaction_Frequency',
    'transaction_hour',
    'is_night_transaction',
    'customer_txn_count',
    'customer_avg_amount',
    'device_txn_count',
    'merchant_txn_count'
]

# 2️⃣ Scale Features (MANDATORY)
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_ann = scaler.fit_transform(train_df[ANN_FEATURES])
X_test_ann  = scaler.transform(test_df[ANN_FEATURES])

y_train_ann = y_train.values
y_test_ann  = y_test.values



In [28]:
# 3️⃣ ANN Architecture (Fraud-Safe)

#  using tensorflow 

# from tensorflow.keras.models import Sequential
# from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
# from tensorflow.keras.optimizers import Adam
# from tensorflow.keras.callbacks import EarlyStopping
# from sklearn.metrics import roc_auc_score, average_precision_score

# model = Sequential([
#     Dense(64, activation='relu', input_shape=(X_train_ann.shape[1],)),
#     BatchNormalization(),
#     Dropout(0.4),

#     Dense(32, activation='relu'),
#     BatchNormalization(),
#     Dropout(0.3),

#     Dense(1, activation='sigmoid')
# ])

# # 4️⃣ Compile (Class Imbalance Aware)
# model.compile(
#     optimizer=Adam(learning_rate=0.001),
#     loss='binary_crossentropy',
#     metrics=['AUC']
# )

# history = model.fit(
#         X_train_ann, y_train_ann,
#         validation_split=0.2,
#         epochs=50,
#         batch_size=256,
#         callbacks=[EarlyStopping(patience=5, restore_best_weights=True)],
#         verbose=0
#     )

# preds = model.predict(X_test_ann).ravel()

import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np

# ---- tensors ----
X_train_t = torch.tensor(X_train_ann, dtype=torch.float32)
y_train_t = torch.tensor(y_train_ann, dtype=torch.float32).view(-1, 1)
X_test_t  = torch.tensor(X_test_ann, dtype=torch.float32)
y_test_t  = torch.tensor(y_test_ann, dtype=torch.float32).view(-1, 1)


# ---- model ----
model = nn.Sequential(
    nn.Linear(X_train_ann.shape[1], 64),
    nn.ReLU(),
    nn.BatchNorm1d(64),
    nn.Dropout(0.4),

    nn.Linear(64, 32),
    nn.ReLU(),
    nn.BatchNorm1d(32),
    nn.Dropout(0.3),

    nn.Linear(32, 1),
    nn.Sigmoid()
)

# ---- optimizer & loss ----
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCELoss()

# ---- train / val split ----
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_t, y_train_t, test_size=0.2, random_state=42
)

# ---- training with early stopping ----
best_loss, patience, counter = np.inf, 5, 0

for _ in range(50):
    model.train()
    optimizer.zero_grad()
    loss = criterion(model(X_tr), y_tr)
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        val_loss = criterion(model(X_val), y_val)

    if val_loss < best_loss:
        best_loss = val_loss
        best_state = model.state_dict()
        counter = 0
    else:
        counter += 1
    if counter >= patience:
        break

model.load_state_dict(best_state)

# ---- predictions ----
model.eval()
with torch.no_grad():
    preds = model(X_test_t).numpy().ravel()

# ---- metrics ----
roc_auc = roc_auc_score(y_test_ann, preds)
pr_auc  = average_precision_score(y_test_ann, preds)

roc_auc, pr_auc


(0.9237132352941176, 0.909262365955718)

In [29]:
import mlflow
import mlflow.pytorch
from mlflow.data import from_numpy
from sklearn.metrics import roc_auc_score, average_precision_score

with mlflow.start_run(run_name="ANN_PyTorch"):

    # --------------------
    # Log dataset lineage
    # --------------------
    train_dataset = from_numpy(
        features=X_train_ann,
        targets=y_train_ann,
        name="fraud_train_ann"
    )

    test_dataset = from_numpy(
        features=X_test_ann,
        targets=y_test_ann,
        name="fraud_test_ann"
    )

    mlflow.log_input(train_dataset, context="training")
    mlflow.log_input(test_dataset, context="testing")

    # --------------------
    # Model parameters
    # --------------------
    mlflow.log_param("model_type", "ANN_PyTorch")
    mlflow.log_param("framework", "pytorch")
    mlflow.log_param("input_dim", X_train_ann.shape[1])

    mlflow.log_param("layer_1_units", 64)
    mlflow.log_param("layer_2_units", 32)
    mlflow.log_param("dropout_1", 0.4)
    mlflow.log_param("dropout_2", 0.3)

    mlflow.log_param("optimizer", "Adam")
    mlflow.log_param("learning_rate", 0.001)
    mlflow.log_param("loss", "BCELoss")
    mlflow.log_param("epochs_max", 50)
    mlflow.log_param("early_stopping_patience", 5)
    mlflow.log_param("batch_norm", True)

    # --------------------
    # Metrics
    # --------------------
    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.log_metric("pr_auc", pr_auc)

    # --------------------
    # Model artifact
    # --------------------
    mlflow.pytorch.log_model(
        model,
        artifact_path="model"
        # registered_model_name="Fraud_ANN_PyTorch"    # this is used to register the model in model registery 
    )


2026/01/15 21:04:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Successfully registered model 'Fraud_ANN_PyTorch'.
2026/01/15 21:05:02 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Fraud_ANN_PyTorch, version 1


🏃 View run ANN_PyTorch at: http://127.0.0.1:5000/#/experiments/415765464730110731/runs/9dc917542f26465fa224a44c711d5f2c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/415765464730110731


Created version '1' of model 'Fraud_ANN_PyTorch'.
